# Sheet 3, Task 8: the errand in description logic

Integrated Intelligent Systems · Lecture 3. This notebook runs the course's teaching ontology of the robot–world manipulation system (every concept under DUL) with the reasoner HermiT. Run the cells from top to bottom.

You need Python 3.10 or newer and Java (HermiT is a Java program); `owlready2` is installed by the first cell if it is missing.

In [ ]:
import shutil, subprocess, sys
from pathlib import Path
print("Python:", sys.version.split()[0])
try:
    import owlready2
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "owlready2"], check=True)
    import owlready2
print("owlready2:", owlready2.VERSION)
java = shutil.which("java")
assert java, "HermiT needs Java. Install a JDK (e.g. 'sudo apt install default-jre') or use the course's Binder lab."
print("Java:", subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0])
# the course ontology lives in ../dl (rwms_ontology.py, DUL.owl, lab_helpers.py)
DL = (Path.cwd() / "../dl").resolve()
sys.path.insert(0, str(DL))
import rwms_ontology as R
from lab_helpers import fresh, consistent, wholes_of
print("ontology:", DL / "rwms_ontology.py")

## 0 · The lecture's competency questions
This rebuilds the ontology from the lecture and lets HermiT answer its nine competency questions. Compare with the slides.

In [ ]:
results = R.run()          # builds the ontology, runs HermiT, answers the nine competency questions of the lecture

## 8.1 · Extend the ontology (3 points)
A `Drawer` is a `Container` with exactly one `Handle` as a component; a `Handle` is a `dul:PhysicalArtifact`; a `CerealBox` is a `TransportableObject` with exactly one mass.

Write the axioms in DL notation in the text cell below, and in code in `extend`. Use `dul:hasComponent` for the count, not the transitive `dul:hasPart`: OWL 2 forbids number restrictions on transitive roles.

*Your axioms in DL notation, each with its kind (partonomy, role restriction, attribute):*

In [ ]:
def extend(o, D):
    """Task 8.1: add Drawer, Handle and CerealBox to the ontology.

    o is the course ontology: o.Container, o.TransportableObject, o.mass, ...
    D is DUL:                 D.PhysicalArtifact, D.hasComponent, ...
    Syntax (owlready2):  class A(B): pass                  A ⊑ B
                         A.is_a.append(r.exactly(1, C))     A ⊑ (=1 r.C)
                         A.is_a.append(r.some(C))           A ⊑ ∃r.C
                         A.is_a.append(r.only(C | E))       A ⊑ ∀r.(C ⊔ E)
    """
    class Handle(D.PhysicalArtifact): pass        # given: Handle ⊑ dul:PhysicalArtifact
    # TODO: Drawer ⊑ Container ⊓ (=1 dul:hasComponent.Handle)
    # TODO: CerealBox ⊑ TransportableObject ⊓ (=1 mass.float)
    # individuals for 8.2 and 8.3 (keep them)
    Knife = type("Knife", (o.WorldEntity,), {})
    Knife("knife")


print("consistent:", consistent(extend))

## 8.2 · Can the knife be a component of the PR2's body? (3 points)
Add the statement and ask the reasoner. If the ontology stays consistent, it does not rule the statement out. Explain why from the axioms.

In [ ]:
def knife_in_body(o, D):
    extend(o, D)
    o["pr2_body"].hasComponent.append(o["knife"])      # the statement to test

print("Can the knife be a component of the PR2's body? Consistent:", consistent(knife_in_body))

*Your explanation:*

Now add the **weakest** axiom that makes the ontology answer *no*, and ask again.

In [ ]:
def knife_in_body_repaired(o, D):
    extend(o, D)
    # TODO: add the weakest axiom about RobotBody that rules the statement out
    o["pr2_body"].hasComponent.append(o["knife"])

print("After the repair, consistent:", consistent(knife_in_body_repaired))

## 8.3 · Of which wholes is the drawer's handle a part? (2 points)
Add the drawer and its handle as individuals, then ask the reasoner. Which axiom produces the answer?

In [ ]:
def with_drawer(o, D):
    extend(o, D)
    # TODO: create drawer5_top (a Drawer) and drawer5_handle (a Handle);
    #       make the handle a component of the drawer and the drawer a component of o["pr2_apartment_bws"]

print(wholes_of("drawer5_handle", ["drawer5_top", "pr2_apartment_bws", "pr2_rwms", "pr2_body"], with_drawer))

*Your answer and the axiom that produces it:*

## 8.4 · What description logic cannot answer (2 points)
Name one competency question about this errand that description logic cannot answer, and say which representation of the design thesis answers it instead.

*Your answer:*